# 14 — Submission de la Base actual (Paso 0 de la última ronda)

La última submission subida es de antes del modelo de texto (≈ 0,92 en el leaderboard público). Esta notebook genera una submission con la **Base del laboratorio** (0,9339 ± 0,0018 en CV) para confirmar que ese número se sostiene en datos que el modelo nunca vio.

No toca `train.ipynb` ni `predict.ipynb`. **No sube nada**: el CSV lo subo a mano.

Pasos:
1. Cargar train y test, y aplicarles **exactamente las mismas funciones** (`aplanar_anidadas`, `agregar_features_contenido`). Son determinísticas, así que dan lo mismo sin importar si se aplican a train o a test.
2. Armar el pipeline de la Base (idéntico a `armar_pipeline()` de `laboratorio.ipynb`, sin la caché) y ajustarlo con **todo** el train.
3. Predecir el test y guardar `submission/submission_base_0934.csv` con el formato confirmado por Kaggle: `ID` = número de fila de 1 a 30.000 (orden de `test_data.jsonlines`) y `condition` = `"new"`/`"used"`.
4. Asserts de formato y dos chequeos de sentido común.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, TargetEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

from features import (
    aplanar_anidadas, agregar_logs, agregar_features_contenido, CapeoPorMoneda,
    VendedorEncoder, PrediccionOOF, armar_modelo_texto, COLUMNAS_CONTENIDO,
)

SEED = 42

## 1. Datos

`y` se arma **antes** de transformar (igual que en todas las notebooks): 1 = `new`, 0 = `used`. Al test se le aplican las mismas dos funciones que al train.

In [2]:
train = pd.read_json("../data/train_data.jsonlines", lines=True)
test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)

train = agregar_features_contenido(aplanar_anidadas(train))
test = agregar_features_contenido(aplanar_anidadas(test))

print("train:", train.shape, "| test:", test.shape)

train: (70000, 72) | test: (30000, 71)


## 2. Pipeline de la Base y ajuste con todo el train

Mismo pipeline que la Base de `laboratorio.ipynb`:
- capeo de precio por moneda → logs;
- one-hot de 4 categóricas, numéricas tal cual (incluye las 6 de `COLUMNAS_CONTENIDO`);
- `TargetEncoder` de `category_id`, TF-IDF de 500 del título;
- `PrediccionOOF(armar_modelo_texto())` sobre título + garantía;
- `VendedorEncoder` con target;
- HistGB fijo.

Al llamar a `.fit` con todo el train, cada transformer aprende con las 70.000 filas. `PrediccionOOF` arma su feature de train con su CV interna y guarda una regresión logística entrenada con todo el train, que es la que va a usar sobre el test. `VendedorEncoder` y `TargetEncoder` hacen lo mismo con sus resguardos (*leave-one-out* y *cross-fitting*).

In [3]:
columnas_categoricas = ["listing_type_id", "buying_mode", "shipping_mode", "seller_state"]
columnas_numericas = [
    "log_price", "log_available_quantity", "automatic_relist",
    "sold_quantity", "initial_quantity",
    "n_fotos", "n_atributos", "n_variaciones", "tiene_video", "tiene_garantia",
    "free_shipping", "local_pick_up", "n_metodos_pago", "accepts_mercadopago",
    "es_tienda_oficial", "tiene_descuento", "tiene_oferta", "tiene_parent_item",
] + COLUMNAS_CONTENIDO

pipeline_base = Pipeline([
    ("capeo", CapeoPorMoneda(columnas=["price", "base_price"], columna_grupo="currency_id")),
    ("logs", FunctionTransformer(agregar_logs)),
    ("preprocesador", ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas),
        ("categoria", TargetEncoder(target_type="binary", random_state=SEED), ["category_id"]),
        ("tfidf", TfidfVectorizer(max_features=500), "title"),
        ("texto", PrediccionOOF(armar_modelo_texto(), random_state=SEED), ["title", "warranty_texto"]),
        ("vendedor", VendedorEncoder(incluir_target=True), ["seller_id", "log_price"]),
    ], sparse_threshold=0)),
    ("modelo", HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED)),
])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pipeline_base.fit(train, y)

print("Árboles usados (early stopping):", pipeline_base.named_steps["modelo"].n_iter_)

Árboles usados (early stopping): 110


## 3. Predicción y CSV

`predict` devuelve 1/0; lo paso a `"new"`/`"used"`. `ID` es el número de fila **empezando en 1**, en el orden del archivo de test (formato confirmado por Kaggle: con ID desde 0 o con el `id` "MLA..." da error).

In [4]:
prediccion = pipeline_base.predict(test)

submission = pd.DataFrame({
    "ID": np.arange(1, len(test) + 1),
    "condition": pd.Series(prediccion).map({1: "new", 0: "used"}),
})

Path("../submission").mkdir(exist_ok=True)
archivo = "../submission/submission_base_0934.csv"
submission.to_csv(archivo, index=False)
print("Guardado:", archivo)
submission.head()

Guardado: ../submission/submission_base_0934.csv


,ID,condition
0,1,used
1,2,new
2,3,used
3,4,used
4,5,new


## 4. Chequeos

**Asserts de formato** (si alguno falla, la celda tira error y el archivo no sirve). Leo el CSV **desde el disco**, para chequear lo que realmente se va a subir:
- 30.000 filas y exactamente las columnas `ID`, `condition`;
- `ID` = 1, 2, …, 30.000 en orden;
- `condition` solo `"new"`/`"used"`, sin nulos.

**Chequeos de sentido común** (no son asserts, son para mirar):
- % de `new` predicho: en train es 53,7%. Si el test es una muestra del mismo universo, debería andar cerca.
- Cuántas predicciones cambian respecto de la submission vieja del HistGB (`submission_histgb.csv`, sin modelo de texto). Si la CV pasó de ~0,924 a ~0,934, deberían cambiar algunos miles de filas, no casi todas.

In [5]:
leida = pd.read_csv(archivo)

assert leida.shape == (30000, 2), leida.shape
assert list(leida.columns) == ["ID", "condition"], list(leida.columns)
assert (leida["ID"].to_numpy() == np.arange(1, 30001)).all()
assert leida["condition"].isin(["new", "used"]).all()
assert leida["condition"].notna().all()
print("Formato OK")

print("% new predicho en test:", round((leida["condition"] == "new").mean(), 4), "| % new en train:", round(y.mean(), 4))

anterior = Path("../submission/submission_histgb.csv")
if anterior.exists():
    vieja = pd.read_csv(anterior)
    cambian = (vieja["condition"] != leida["condition"]).sum()
    print("Predicciones distintas vs submission_histgb.csv:", cambian, f"({cambian / len(leida):.1%})")

Formato OK
% new predicho en test: 0.5287 | % new en train: 0.537
Predicciones distintas vs submission_histgb.csv: 1367 (4.6%)
